# Automated HTML Email Sending

This notebook reads contacts from `input.csv`, personalises the greeting in the existing HTML template, and sends each recipient an individualised email while reporting success or failure per row.

In [5]:
# Imports for data handling and email composition.
import os
from pathlib import Path
from typing import Dict, List

import pandas as pd

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage

In [6]:
# Load the contact list exported from the spreadsheet and normalise the fields.
contacts_path = Path('input.csv')

contacts_df = (
    pd.read_csv(contacts_path)
    .rename(columns=str.strip)
    .assign(
        Name=lambda df: df['Name'].astype(str).str.strip(),
        Email=lambda df: df['Email'].astype(str).str.strip(),
    )
)

# Remove any rows without an address so we do not attempt to send incomplete messages.
contacts_df = contacts_df.loc[contacts_df['Email'].ne('')].reset_index(drop=True)
contacts_df

,Name,Email
0,Sabrina's iCloud,sdb5s@icloud.com
1,Sabrina's Scarletmail,sb2071@scarletmail.rutgers.edu


In [ ]:
# Prepare the HTML template and the inline image required by the message.
template_path = Path('test.html')
raw_template = template_path.read_text(encoding='utf-8')

greeting_token = 'Hi&nbsp;<em>Maine News Online</em>&nbsp;editor,'
if greeting_token not in raw_template:
    raise ValueError('Expected greeting token not found in HTML template.')

# Replace the static greeting with a placeholder the script can fill per recipient.
email_template_html = raw_template.replace(greeting_token, 'Hi&nbsp;<em>{{name}}</em>,')
assert '{{name}}' in email_template_html, 'Name placeholder missing in template.'

image_path = Path('picture.jpeg')
if not image_path.exists():
    raise FileNotFoundError('Missing inline image at picture.jpeg referenced in the HTML template.')

image_content_id = image_path.name
print('HTML template and inline image loaded.')

In [7]:
# Collect SMTP connection settings from environment variables to avoid hardcoding secrets.
def load_smtp_config() -> Dict[str, object]:
    config = {
        'host': os.environ.get('SMTP_HOST'),
        'port': int(os.environ.get('SMTP_PORT', '587')),
        'username': os.environ.get('SMTP_USERNAME'),
        'password': os.environ.get('SMTP_PASSWORD'),
        'sender': os.environ.get('SMTP_SENDER') or os.environ.get('SMTP_USERNAME'),
        'use_tls': os.environ.get('SMTP_USE_TLS', 'true').lower() in {'1', 'true', 'yes'},
        'use_ssl': os.environ.get('SMTP_USE_SSL', 'false').lower() in {'1', 'true', 'yes'},
    }
    if config['use_ssl'] and config['use_tls']:
        raise ValueError('Choose either TLS or SSL, not both.')
    missing = [key for key in ('host', 'port', 'username', 'password', 'sender') if not config[key]]
    if missing:
        raise ValueError('Missing SMTP configuration values: ' + ', '.join(missing))
    return config

try:
    SMTP_CONFIG = load_smtp_config()
    print('SMTP configuration loaded successfully.')
except ValueError as exc:
    SMTP_CONFIG = None
    print(f'SMTP configuration issue: {exc}')

EMAIL_SUBJECT = os.environ.get('EMAIL_SUBJECT', 'THE GLASS EEL - Author Events in Maine')
REPLY_TO = os.environ.get('EMAIL_REPLY_TO')

SMTP configuration issue: Missing SMTP configuration values: host, username, password, sender


In [ ]:
# Create the MIME message for a single contact, including plain-text fallback and inline image.
def build_email_message(contact: Dict[str, str], html_template: str, config: Dict[str, object]):
    name = (contact.get('Name') or '').strip()
    email = (contact.get('Email') or '').strip()
    if not email:
        raise ValueError('Contact is missing an email address.')

    display_name = name or 'there'

    message = MIMEMultipart('related')
    message['Subject'] = EMAIL_SUBJECT
    message['From'] = config['sender']
    message['To'] = email
    if REPLY_TO:
        message['Reply-To'] = REPLY_TO

    alternative = MIMEMultipart('alternative')
    message.attach(alternative)

    plain_body = f"Hi {display_name},\nPlease view this announcement in HTML format."
    alternative.attach(MIMEText(plain_body, 'plain', 'utf-8'))

    html_body = html_template.replace('{{name}}', display_name)
    alternative.attach(MIMEText(html_body, 'html', 'utf-8'))

    with image_path.open('rb') as img_file:
        image_part = MIMEImage(img_file.read())
    image_part.add_header('Content-ID', f'<{image_content_id}>')
    image_part.add_header('Content-Disposition', 'inline', filename=image_path.name)
    message.attach(image_part)

    return message, email, display_name

In [ ]:
# Send personalised emails to every contact and report progress row by row.
def send_personalised_emails(contacts: List[Dict[str, str]], html_template: str, config: Dict[str, object]):
    if config is None:
        raise RuntimeError('SMTP settings are not configured. Set the environment variables and rerun the configuration cell.')

    results = []
    smtp_class = smtplib.SMTP_SSL if config['use_ssl'] else smtplib.SMTP

    with smtp_class(config['host'], config['port']) as server:
        server.ehlo()
        if config['use_tls'] and not config['use_ssl']:
            server.starttls()
            server.ehlo()
        server.login(config['username'], config['password'])

        for contact in contacts:
            try:
                message, recipient_email, display_name = build_email_message(contact, html_template, config)
                server.sendmail(config['sender'], [recipient_email], message.as_string())
                print(f'[OK] {display_name} <{recipient_email}>')
                results.append({'name': display_name, 'email': recipient_email, 'status': 'sent'})
            except Exception as exc:
                name = (contact.get('Name') or '').strip() or 'Unknown recipient'
                email = (contact.get('Email') or '').strip()
                print(f'[FAIL] {name} <{email}> :: {exc}')
                results.append({'name': name, 'email': email, 'status': f'error: {exc}'})
    return results

send_results = send_personalised_emails(contacts_df.to_dict(orient='records'), email_template_html, SMTP_CONFIG)
send_results